# 16 · Structured block pruning

**Goal:** remove each repeated 5×5, 128-channel depthwise-separable block independently, recover the best candidate with EMA-range QAT, export full INT8, and hand the exact FlatBuffer to the ESP32 profiler. No layer is masked: the selected block is absent from the deployed graph.

## Evidence flow

`Fast-80 weights → remove block7/block8/block9 → short equal-budget screen → select by validation PR-AUC → EMA-QAT recovery → locked test → full-INT8 export → physical ESP32 profile`

The three blocks have identical input/output shapes, so this experiment isolates *where* one late block can be removed without simultaneously changing widths.

In [ ]:
import hashlib, json, subprocess, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display

from vww_esp32.config import load_config, seed_everything
from vww_esp32.evaluation import classification_metrics, collect_predictions, expected_calibration_error, select_threshold
from vww_esp32.exporting import convert_full_integer, inspect_tflite, run_tflite
from vww_esp32.modeling import build_tiny_mobilenet_v1, compile_model
from vww_esp32.preprocessing import make_dataset, representative_dataset, split_manifest
from vww_esp32.profiling import profile_model
from vww_esp32.pruning import block_pruning_specs, transfer_structured_weights
from vww_esp32.qat import build_qat_mobilenet_v1, copy_compatible_weights, copy_qat_weights_to_float

config, ROOT = load_config('configs/pruning.yaml')
seed = int(config['project']['seed']); seed_everything(seed)
IMAGE_SIZE = tuple(config['preprocessing']['image_size'])
BATCH_SIZE = int(config['preprocessing']['batch_size'])
OUT = ROOT / 'artifacts/pruning/block'
for child in ('models', 'reports', 'figures', 'logs'): (OUT / child).mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
print(f'TensorFlow {tf.__version__} · root={ROOT}')

## 1 · Freeze data and reference evidence

All candidates use the existing train/validation/test manifests. Selection uses validation PR-AUC only; the held-out test set is opened once for the recovered winner.

In [ ]:
manifest = pd.read_csv(ROOT / 'data/processed/manifest.csv')
splits = split_manifest(manifest)
assert {key: len(value) for key, value in splits.items()} == {'train': 12000, 'val': 2000, 'test': 2000}
assert sum((~frame.image_path.map(Path.exists)).sum() for frame in splits.values()) == 0
train_ds = make_dataset(splits['train'], IMAGE_SIZE, BATCH_SIZE, training=True, seed=seed)
val_ds = make_dataset(splits['val'], IMAGE_SIZE, BATCH_SIZE, training=False)
test_ds = make_dataset(splits['test'], IMAGE_SIZE, BATCH_SIZE, training=False)
reference = tf.keras.models.load_model(ROOT / config['reference']['float_model'])
reference_metrics = json.loads((ROOT / config['reference']['int8_metrics']).read_text())
_, _, reference_profile = profile_model(reference, training_batch_size=BATCH_SIZE)
display(pd.DataFrame([{'split': key, 'images': len(value), 'person_rate': value.label.mean()} for key, value in splits.items()]))

## 2 · Physically rebuild three block ablations

Surviving layers keep their Fast-80 tensors. Each target is audited to prove that six Keras layers (depthwise, BatchNorm, ReLU6, pointwise, BatchNorm, ReLU6) disappeared and that parameters/MACs decreased.

In [ ]:
specs = {spec.name: spec for spec in block_pruning_specs()}
candidates, structural_rows = {}, []
for name, spec in specs.items():
    model = build_tiny_mobilenet_v1(input_shape=(*IMAGE_SIZE, 3), dropout=config['model']['dropout'], l2=config['model']['l2'], channel_schedule=spec.channel_schedule, removed_blocks=spec.removed_blocks)
    transfer = transfer_structured_weights(reference, model, spec.channel_schedule, spec.removed_blocks, method=spec.ranking)
    layers, liveness, profile = profile_model(model, training_batch_size=BATCH_SIZE)
    assert profile['estimated_macs_batch1'] < reference_profile['estimated_macs_batch1']
    candidates[name] = model
    structural_rows.append({'candidate': name, 'removed': spec.removed_blocks[0], 'parameters': profile['parameters'], 'macs': profile['estimated_macs_batch1'], 'mac_reduction_pct': 100 * (1 - profile['estimated_macs_batch1'] / reference_profile['estimated_macs_batch1']), 'peak_live_int8': profile['peak_live_activation_int8_bytes_batch1_hypothetical'], 'layers': profile['layers']})
    (OUT / 'reports' / f'{name}_transfer.json').write_text(json.dumps(transfer, indent=2) + '\n')
structure = pd.DataFrame(structural_rows)
structure.to_csv(OUT / 'reports/structural_screen.csv', index=False)
display(structure)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=structure, x='candidate', y='macs', ax=axes[0], color='#21C7E8'); axes[0].tick_params(axis='x', rotation=20); axes[0].set_title('Physical compute after block removal')
sns.barplot(data=structure, x='candidate', y='parameters', ax=axes[1], color='#F2994A'); axes[1].tick_params(axis='x', rotation=20); axes[1].set_title('Stored parameters after block removal')
fig.tight_layout(); fig.savefig(OUT / 'figures/block_structure.png', dpi=170, bbox_inches='tight'); plt.show()

## 3 · Equal-budget float screening

Every ablation receives exactly the same short recovery budget. Validation PR-AUC is the primary selector, with recall and F1 reported as safety signals. This screen avoids spending full QAT time on all three blocks.

In [ ]:
screen_rows = []
for name, model in candidates.items():
    compile_model(model, learning_rate=float(config['training']['learning_rate']), label_smoothing=float(config['training']['label_smoothing']))
    started = time.monotonic()
    history = model.fit(train_ds, validation_data=val_ds, epochs=int(config['training']['screening_epochs']), verbose=1)
    y_val, p_val = collect_predictions(model, val_ds)
    choice = select_threshold(y_val, p_val, objective=config['evaluation']['threshold_objective'], minimum_recall=float(config['evaluation']['minimum_recall']))
    metrics = classification_metrics(y_val, p_val, choice['threshold'])
    metrics.update({'candidate': name, 'seconds': time.monotonic() - started, 'best_epoch_pr_auc': max(history.history['val_pr_auc'])})
    screen_rows.append(metrics)
    model.save(OUT / 'models' / f'{name}_screened.keras')
screen = pd.DataFrame(screen_rows).sort_values(['pr_auc', 'f1'], ascending=False)
screen.to_csv(OUT / 'reports/validation_screen.csv', index=False)
winner_name = screen.iloc[0].candidate; winner_spec = specs[winner_name]; winner_float = candidates[winner_name]
display(screen[['candidate', 'threshold', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'pr_auc', 'seconds']])
print('Selected by validation PR-AUC:', winner_name)

## 4 · EMA-range QAT recovery for the winner

The selected QAT recipe is applied only after structural selection. BatchNorm is frozen, activation observers warm on training batches, and the training-only quantizers are stripped before export.

In [ ]:
qat_model = build_qat_mobilenet_v1(input_shape=(*IMAGE_SIZE, 3), dropout=config['model']['dropout'], l2=config['model']['l2'], range_mode=config['qat']['range_mode'], ema_decay=float(config['qat']['ema_decay']), channel_schedule=winner_spec.channel_schedule, removed_blocks=winner_spec.removed_blocks)
transfer = copy_compatible_weights(winner_float, qat_model)
for layer in qat_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization): layer.trainable = False
for images, _ in train_ds.take(64): _ = qat_model(images, training=True)
compile_model(qat_model, learning_rate=float(config['training']['qat_learning_rate']), label_smoothing=0.0)
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=int(config['training']['early_stopping_patience']), restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max', factor=0.3, patience=int(config['training']['reduce_lr_patience']), min_lr=1e-7)]
history = qat_model.fit(train_ds, validation_data=val_ds, epochs=int(config['training']['recovery_epochs']), callbacks=callbacks, verbose=1)
deployment_model = build_tiny_mobilenet_v1(input_shape=(*IMAGE_SIZE, 3), dropout=config['model']['dropout'], l2=config['model']['l2'], channel_schedule=winner_spec.channel_schedule, removed_blocks=winner_spec.removed_blocks)
strip = copy_qat_weights_to_float(qat_model, deployment_model)
assert not strip['skipped_layers'], strip
deployment_model.save(OUT / 'models/block_pruned_qat_float.keras')
pd.DataFrame(history.history).to_csv(OUT / 'logs/qat_history.csv', index=False)

## 5 · Locked evaluation, full-INT8 export, and resource gates

The decision threshold is selected on validation data and locked. Test data is used only for the final float/INT8 report. Structural and quality gates are recorded separately so an accurate but non-smaller graph cannot pass as pruning.

In [ ]:
y_val, p_val = collect_predictions(deployment_model, val_ds)
threshold = select_threshold(y_val, p_val, objective=config['evaluation']['threshold_objective'], minimum_recall=float(config['evaluation']['minimum_recall']))['threshold']
y_test, p_float = collect_predictions(deployment_model, test_ds)
float_metrics = classification_metrics(y_test, p_float, threshold)
model_path = OUT / 'models/vww_block_pruned_qat_int8.tflite'
representative = representative_dataset(splits['train'], IMAGE_SIZE, int(config['export']['representative_samples']), seed)
convert_full_integer(deployment_model, representative, model_path)
contract = inspect_tflite(model_path)
test_images = np.concatenate([np.asarray(images) for images, _ in test_ds])
p_int8 = run_tflite(model_path, test_images)
int8_metrics = classification_metrics(y_test, p_int8, threshold)
int8_metrics['expected_calibration_error_10_bins'] = expected_calibration_error(y_test, p_int8)
int8_metrics['probability_mae_vs_float'] = float(np.mean(np.abs(p_int8 - p_float)))
layers, liveness, model_profile = profile_model(deployment_model, training_batch_size=BATCH_SIZE)
gates = {'full_int8': contract['input']['dtype'] == contract['output']['dtype'] == 'int8', 'physically_fewer_macs': model_profile['estimated_macs_batch1'] < reference_profile['estimated_macs_batch1'], 'recall': int8_metrics['recall'] >= config['deployment_gate']['minimum_test_recall'], 'f1': int8_metrics['f1'] >= config['deployment_gate']['minimum_test_f1'], 'model_bytes': model_path.stat().st_size <= config['deployment_gate']['max_model_bytes']}
summary = {'experiment': 'structured_block_pruning', 'selected_candidate': winner_name, 'spec': winner_spec.as_dict(), 'threshold': threshold, 'float_test_metrics': float_metrics, 'int8_test_metrics': int8_metrics, 'model_profile': model_profile, 'export': contract, 'gates': gates, 'ready_for_device_profile': all(gates.values()), 'model_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest()}
(OUT / 'reports/experiment_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
layers.to_csv(OUT / 'reports/layer_profile.csv', index=False); liveness.to_csv(OUT / 'reports/activation_liveness.csv', index=False)
display(pd.DataFrame([reference_metrics, int8_metrics], index=['Fast-80 PTQ', 'block-pruned QAT'])[['accuracy', 'precision', 'recall', 'specificity', 'f1', 'pr_auc']])
display(pd.DataFrame({'gate': gates.keys(), 'passed': gates.values()})); print(model_profile)

## 6 · Explicit physical-device handoff

Analytical MAC reduction is not accepted as a speed result. Run the command below only if every offline gate passes, then compare Invoke, arena use, operator latency, and end-to-end FPS with the same-session Fast-80 control.

In [ ]:
SERIAL_PORT = '/dev/cu.YOUR_SERIAL_PORT'
command = [str(ROOT / 'scripts/profile_esp32_model.sh'), str(model_path), '80', 'prune_block_selected', SERIAL_PORT]
print(' '.join(command))
RUN_DEVICE_PROFILE = False
if RUN_DEVICE_PROFILE:
    assert summary['ready_for_device_profile'] and 'YOUR_SERIAL_PORT' not in SERIAL_PORT
    subprocess.run(command, cwd=ROOT, check=True)